# Context Bank Diagnostics

Use this notebook after a `build_contextual_bank.py` run to inspect token-count coverage, active cluster usage, cluster occupancy balance, and simple heuristics for choosing `codebook_k` and `codebook_min_count`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'build_contextual_bank.py').exists():
            return candidate
    return start.resolve()


REPO_ROOT = find_repo_root(Path.cwd())
ARTIFACT_DIR = REPO_ROOT / 'banks/context_bank_capybara_debug'
SELECTED_DISTRIBUTION = None
SELECTED_LAYER = None

print(f'Repo root: {REPO_ROOT}')
print(f'Artifact dir: {ARTIFACT_DIR}')

Repo root: /home/clint/workspace/projects/disentangle_probe
Artifact dir: /home/clint/workspace/projects/disentangle_probe/banks/context_bank_capybara_debug


In [2]:
def load_meta(artifact_dir: Path) -> dict:
    meta_path = artifact_dir / 'meta.json'
    if not meta_path.exists():
        raise FileNotFoundError(f'Metadata not found at {meta_path}')
    return json.loads(meta_path.read_text(encoding='utf-8'))


def available_distributions(artifact_dir: Path) -> list[str]:
    return sorted(
        path.name.removeprefix('distribution_')
        for path in artifact_dir.glob('distribution_*')
        if path.is_dir()
    )


def choose_distribution(artifact_dir: Path, requested: str | None) -> str:
    distributions = available_distributions(artifact_dir)
    if not distributions:
        raise ValueError(f'No distribution_* directories found under {artifact_dir}')
    if requested is None:
        return distributions[0]
    if requested not in distributions:
        raise ValueError(f'Distribution {requested!r} not found. Available: {distributions}')
    return requested


def choose_layer(meta: dict, requested: int | None) -> int:
    layers = [int(layer) for layer in meta['layers']]
    if requested is None:
        return layers[-1]
    if requested not in layers:
        raise ValueError(f'Layer {requested} not found. Available: {layers}')
    return int(requested)


def load_distribution_layer(artifact_dir: Path, distribution: str, layer: int) -> tuple[dict, dict]:
    base = artifact_dir / f'distribution_{distribution}'
    stats = torch.load(base / f'layer_{layer}_stats.pt', map_location='cpu')
    codebook = torch.load(base / f'layer_{layer}_codebook.pt', map_location='cpu')
    return stats, codebook


def compute_centroid_spread(centroids: torch.Tensor, active_clusters: int) -> float:
    active = centroids[:active_clusters].to(torch.float32)
    if active_clusters <= 1:
        return 0.0
    distances = torch.cdist(active, active, p=2)
    upper = distances[torch.triu(torch.ones_like(distances, dtype=torch.bool), diagonal=1)]
    if upper.numel() == 0:
        return 0.0
    return float(upper.mean().item())


def build_layer_dataframe(stats: dict, codebook: dict) -> pd.DataFrame:
    token_ids = stats['token_ids'].to(torch.long)
    counts = stats['counts'].to(torch.long)
    active_clusters = codebook['active_clusters'].to(torch.long)
    cluster_counts = codebook['cluster_counts'].to(torch.long)
    mean = stats['mean'].to(torch.float32)
    variance = stats['variance'].to(torch.float32)
    centroids = codebook['centroids'].to(torch.float32)
    cluster_variance = codebook['cluster_variance'].to(torch.float32)

    rows = []
    for idx in range(len(token_ids)):
        token_id = int(token_ids[idx].item())
        count = int(counts[idx].item())
        active = int(active_clusters[idx].item())
        counts_row = cluster_counts[idx, : max(active, 1)].numpy()
        total_cluster_count = int(cluster_counts[idx].sum().item())
        normalized_counts = counts_row / max(counts_row.sum(), 1)
        nonzero = normalized_counts[normalized_counts > 0]
        entropy = float(-(nonzero * np.log(nonzero)).sum()) if len(nonzero) else 0.0
        max_share = float(normalized_counts.max()) if len(normalized_counts) else 0.0
        sorted_shares = np.sort(normalized_counts)[::-1] if len(normalized_counts) else np.array([0.0])
        second_share = float(sorted_shares[1]) if len(sorted_shares) > 1 else 0.0
        tiny_share = float((counts_row <= 1).sum() / max(active, 1)) if active > 0 else 0.0
        centroid_spread = compute_centroid_spread(centroids[idx], active)
        mean_variance_norm = float(torch.linalg.vector_norm(variance[idx]).item())
        cluster_variance_norm = float(torch.linalg.vector_norm(cluster_variance[idx, : max(active, 1)]).item())
        mean_norm = float(torch.linalg.vector_norm(mean[idx]).item())
        rows.append({
            'token_id': token_id,
            'count': count,
            'active_clusters': active,
            'total_cluster_count': total_cluster_count,
            'max_cluster_share': max_share,
            'second_cluster_share': second_share,
            'tiny_cluster_fraction': tiny_share,
            'cluster_entropy': entropy,
            'centroid_spread': centroid_spread,
            'variance_norm': mean_variance_norm,
            'cluster_variance_norm': cluster_variance_norm,
            'mean_norm': mean_norm,
        })
    return pd.DataFrame(rows).sort_values(['count', 'token_id'], ascending=[False, True]).reset_index(drop=True)


def summarize_recommendations(df: pd.DataFrame, meta: dict) -> pd.Series:
    if df.empty:
        return pd.Series({'note': 'No tokens found'})

    configured_k = int(meta['codebook_k'])
    configured_min_count = int(meta['codebook_min_count'])
    active_hist = df['active_clusters'].value_counts().sort_index()
    saturated_rate = float((df['active_clusters'] >= configured_k).mean())
    multi_cluster_rate = float((df['active_clusters'] >= 2).mean())
    low_support_rate = float((df['count'] < configured_min_count).mean())
    median_count = float(df['count'].median())
    p75_count = float(df['count'].quantile(0.75))
    p90_count = float(df['count'].quantile(0.90))

    suggested_k = configured_k
    if saturated_rate > 0.15:
        suggested_k = configured_k * 2
    elif float((df['active_clusters'] <= 2).mean()) > 0.95 and configured_k > 4:
        suggested_k = max(4, configured_k // 2)

    suggested_min_count = configured_min_count
    if low_support_rate > 0.50:
        suggested_min_count = max(configured_k, int(max(4, round(p75_count))))
    elif float((df['tiny_cluster_fraction'] > 0).mean()) > 0.25:
        suggested_min_count = max(configured_min_count, configured_k * 2)

    return pd.Series({
        'configured_k': configured_k,
        'configured_min_count': configured_min_count,
        'num_tokens': int(len(df)),
        'multi_cluster_rate': multi_cluster_rate,
        'saturated_rate': saturated_rate,
        'low_support_rate': low_support_rate,
        'median_count': median_count,
        'p75_count': p75_count,
        'p90_count': p90_count,
        'suggested_k': int(suggested_k),
        'suggested_min_count': int(suggested_min_count),
        'active_cluster_histogram': dict(active_hist),
    })


In [3]:
meta = load_meta(ARTIFACT_DIR)
distribution = choose_distribution(ARTIFACT_DIR, SELECTED_DISTRIBUTION)
layer = choose_layer(meta, SELECTED_LAYER)
stats, codebook = load_distribution_layer(ARTIFACT_DIR, distribution, layer)
df = build_layer_dataframe(stats, codebook)

overview = pd.Series({
    'artifact_dir': str(ARTIFACT_DIR),
    'dataset_source': meta.get('dataset_source'),
    'dataset_name': meta.get('dataset_name'),
    'dataset_split': meta.get('dataset_split'),
    'distribution': distribution,
    'layer': layer,
    'available_layers': meta['layers'],
    'available_distributions': available_distributions(ARTIFACT_DIR),
    'num_examples_processed': meta.get('num_examples_processed'),
    'codebook_k': meta.get('codebook_k'),
    'codebook_min_count': meta.get('codebook_min_count'),
    'num_observed_tokens': len(df),
})

display(overview.to_frame('value'))
display(df.head(20))

,value
artifact_dir,/home/clint/workspace/projects/disentangle_pro...
dataset_source,huggingface
dataset_name,HuggingFaceH4/capybara
dataset_split,train_sft
distribution,content_only
layer,8
available_layers,"[0, 1, 8]"
available_distributions,[content_only]
num_examples_processed,128
codebook_k,4


,token_id,count,active_clusters,total_cluster_count,max_cluster_share,second_cluster_share,tiny_cluster_fraction,cluster_entropy,centroid_spread,variance_norm,cluster_variance_norm,mean_norm
0,11,2253,4,2253,0.652907,0.234354,0.00,0.873581,0.420367,0.013767,0.020515,0.826945
1,279,1920,4,1920,0.496354,0.320833,0.25,1.026638,0.479343,0.014618,0.021479,0.800692
2,13,1471,4,1471,0.475187,0.345343,0.00,1.118068,0.406761,0.016679,0.033021,0.854788
3,323,1261,4,1261,0.648692,0.172086,0.00,1.011130,0.367206,0.011351,0.019017,0.851117
4,315,1077,4,1077,0.988858,0.004643,0.00,0.072642,0.635755,0.018703,0.020738,0.731020
5,264,930,4,930,0.795699,0.195699,0.00,0.546811,0.650028,0.014813,0.020683,0.786480
6,311,898,4,898,0.995546,0.002227,0.50,0.033191,0.843537,0.016579,0.017627,0.758348
7,220,706,4,706,0.847025,0.143059,0.25,0.468617,0.684085,0.014287,0.017264,0.797579
8,304,621,4,621,0.942029,0.049919,0.25,0.248736,0.615152,0.013665,0.018696,0.806347
9,16,529,4,529,0.695652,0.134216,0.00,0.915765,0.706492,0.019556,0.022325,0.749545


In [4]:
fig = px.histogram(df, x='count', nbins=60, title=f'Token count histogram: {distribution}, layer {layer}', log_y=True)
fig.update_layout(bargap=0.05)
fig.show()

active_hist = df['active_clusters'].value_counts().sort_index().rename_axis('active_clusters').reset_index(name='tokens')
fig = px.bar(active_hist, x='active_clusters', y='tokens', title=f'Active cluster histogram: {distribution}, layer {layer}')
fig.show()

In [5]:
top_df = df.head(min(200, len(df))).copy()
fig = px.scatter(
    top_df,
    x='count',
    y='active_clusters',
    color='max_cluster_share',
    hover_data=['token_id', 'second_cluster_share', 'tiny_cluster_fraction', 'centroid_spread', 'variance_norm'],
    title=f'Top-token occupancy balance: {distribution}, layer {layer}'
)
fig.show()

spread_df = top_df[top_df['active_clusters'] >= 2].copy()
if not spread_df.empty:
    spread_df['spread_to_variance'] = spread_df['centroid_spread'] / spread_df['cluster_variance_norm'].replace(0, np.nan)
    fig = px.scatter(
        spread_df,
        x='centroid_spread',
        y='cluster_variance_norm',
        size='count',
        hover_data=['token_id', 'active_clusters', 'max_cluster_share', 'second_cluster_share'],
        title=f'Centroid spread vs within-cluster variance: {distribution}, layer {layer}'
    )
    fig.show()
else:
    print('No multi-cluster tokens in this layer/distribution selection.')

In [6]:
recommendation = summarize_recommendations(df, meta)
display(recommendation.to_frame('value'))

frequent_multicluster = df[(df['count'] >= meta['codebook_min_count']) & (df['active_clusters'] >= 2)].copy()
display(frequent_multicluster.head(30))

,value
configured_k,4
configured_min_count,4
num_tokens,8488
multi_cluster_rate,0.251885
saturated_rate,0.251296
low_support_rate,0.748115
median_count,2.0
p75_count,4.0
p90_count,8.0
suggested_k,8


,token_id,count,active_clusters,total_cluster_count,max_cluster_share,second_cluster_share,tiny_cluster_fraction,cluster_entropy,centroid_spread,variance_norm,cluster_variance_norm,mean_norm
0,11,2253,4,2253,0.652907,0.234354,0.00,0.873581,0.420367,0.013767,0.020515,0.826945
1,279,1920,4,1920,0.496354,0.320833,0.25,1.026638,0.479343,0.014618,0.021479,0.800692
2,13,1471,4,1471,0.475187,0.345343,0.00,1.118068,0.406761,0.016679,0.033021,0.854788
3,323,1261,4,1261,0.648692,0.172086,0.00,1.011130,0.367206,0.011351,0.019017,0.851117
4,315,1077,4,1077,0.988858,0.004643,0.00,0.072642,0.635755,0.018703,0.020738,0.731020
5,264,930,4,930,0.795699,0.195699,0.00,0.546811,0.650028,0.014813,0.020683,0.786480
6,311,898,4,898,0.995546,0.002227,0.50,0.033191,0.843537,0.016579,0.017627,0.758348
7,220,706,4,706,0.847025,0.143059,0.25,0.468617,0.684085,0.014287,0.017264,0.797579
8,304,621,4,621,0.942029,0.049919,0.25,0.248736,0.615152,0.013665,0.018696,0.806347
9,16,529,4,529,0.695652,0.134216,0.00,0.915765,0.706492,0.019556,0.022325,0.749545


## How to read the diagnostics

- If most tokens stay at `active_clusters == 1`, the current `codebook_k` is probably larger than you need.
- If many frequent tokens hit `active_clusters == codebook_k`, increase `codebook_k` and rerun.
- If many tokens above `codebook_min_count` still have tiny one-example clusters, raise `codebook_min_count`.
- If `centroid_spread` is small relative to `cluster_variance_norm`, the extra clusters are likely weak or redundant.
- For a quick comparison across layers, rerun the notebook with a different `SELECTED_LAYER` or duplicate the loading cell into a loop over `meta['layers']`.